In [1]:
import pandas as pd
import numpy as np
import pybaseball
from tqdm import tqdm
from pitch_data import get_transformed_data

    
pybaseball.cache.enable()

In [2]:
# def get_pitcher_historical_stats(pitchers):
#     data = []
#     for year in range(2018, 2025):
#         data_pull = statcast_pitcher_percentile_ranks(year)
#         data_pull = data_pull.dropna(axis=0, thresh=len(data_pull.columns) - 1).dropna(axis=1)
#         data.append(data_pull)
    
#     data = pd.concat(data, ignore_index=True)
    
#     # Get meaningful column names (exclude metadata columns)
#     meaningful_cols = [col for col in data.columns if col not in ['player_name', 'player_id', 'year']]
    
#     pitch_stats = {}
#     for pitcher in tqdm(pitchers):
#         if pitcher in data['player_id'].unique():
#             # Get latest stats for this pitcher
#             latest_stats = data[data['player_id'] == pitcher].sort_values('year').iloc[-1]
#             # Extract just the meaningful stat columns
#             pitch_stats[pitcher] = latest_stats[meaningful_cols]
    
#     # Return DataFrame with meaningful column names preserved
#     result_df = pd.DataFrame.from_dict(pitch_stats, orient='index')
#     result_df.columns = meaningful_cols  # Ensure column names are preserved
#     return result_df

# def get_batter_historical_stats(batters):
#     data = []
#     for year in range(2018, 2025):
#         data_pull = statcast_batter_percentile_ranks(year)
#         data_pull = data_pull.dropna(axis=0, thresh=len(data_pull.columns) - 1).dropna(axis=1)
#         data.append(data_pull)
    
#     data = pd.concat(data, ignore_index=True)
    
#     # Get meaningful column names (exclude metadata columns)
#     meaningful_cols = [col for col in data.columns if col not in ['player_name', 'player_id', 'year']]
    
#     batter_stats = {}
#     for batter in tqdm(batters):
#         if batter in data['player_id'].unique():
#             # Get latest stats for this batter
#             latest_stats = data[data['player_id'] == batter].sort_values('year').iloc[-1]
#             # Extract just the meaningful stat columns
#             batter_stats[batter] = latest_stats[meaningful_cols]
    
#     # Return DataFrame with meaningful column names preserved
#     result_df = pd.DataFrame.from_dict(batter_stats, orient='index')
#     result_df.columns = meaningful_cols  # Ensure column names are preserved
#     return result_df

In [3]:
# def get_team_records_data(games_df):
#     """
#     Get team records (wins/losses) up to each game date using ESPN API.
    
#     Parameters:
#     - games_df: DataFrame with columns ['home_team', 'away_team', 'game_date']
    
#     Returns:
#     - DataFrame with team records added as new columns
#     """
#     import requests
#     import json
    
#     # Get unique years from the games
#     years = sorted(games_df['game_date'].dt.year.unique())
    
#     # ESPN team name mapping (ESPN uses different abbreviations)
#     espn_team_mapping = {
#         'ARI': 'AZ', 'ATL': 'ATL', 'BAL': 'BAL', 'BOS': 'BOS', 'CHC': 'CHC',
#         'CWS': 'CWS', 'CIN': 'CIN', 'CLE': 'CLE', 'COL': 'COL', 'DET': 'DET',
#         'HOU': 'HOU', 'KC': 'KC', 'LAA': 'LAA', 'LAD': 'LAD', 'MIA': 'MIA',
#         'MIL': 'MIL', 'MIN': 'MIN', 'NYM': 'NYM', 'NYY': 'NYY', 'OAK': 'OAK',
#         'PHI': 'PHI', 'PIT': 'PIT', 'SD': 'SD', 'SF': 'SF', 'SEA': 'SEA',
#         'STL': 'STL', 'TB': 'TB', 'TEX': 'TEX', 'TOR': 'TOR', 'WSH': 'WSH'
#     }
    
#     def get_standings_from_espn(year):
#         """Get MLB standings from ESPN API for a given year"""
#         try:
#             # ESPN MLB standings API endpoint
#             url = f"https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/scoreboard"
#             params = {
#                 'dates': f"{year}0401-{year}1031",  # April to October
#                 'seasontype': 2  # Regular season
#             }
            
#             # Try to get current season standings
#             standings_url = f"https://site.api.espn.com/apis/v2/sports/baseball/mlb/standings?season={year}"
            
#             response = requests.get(standings_url, timeout=10)
#             response.raise_for_status()
            
#             standings_data = response.json()
            
#             # Parse the standings data
#             teams_records = {}
            
#             if 'children' in standings_data:
#                 # Navigate through the ESPN API structure
#                 for conference in standings_data['children']:  # AL/NL
#                     if 'standings' in conference:
#                         for division in conference['standings']['entries']:
#                             team = division['team']
#                             stats = division['stats']
                            
#                             # Extract team abbreviation and record
#                             team_abbr = team['abbreviation']
                            
#                             # Find wins and losses in stats
#                             wins = losses = 0
#                             for stat in stats:
#                                 if stat['name'] == 'wins':
#                                     wins = stat['value']
#                                 elif stat['name'] == 'losses':
#                                     losses = stat['value']
                            
#                             teams_records[team_abbr] = {
#                                 'wins': wins,
#                                 'losses': losses,
#                                 'win_pct': wins / (wins + losses) if (wins + losses) > 0 else 0.5
#                             }
            
#             if teams_records:
#                 print(f"✅ Loaded {len(teams_records)} teams from ESPN for {year}")
#                 return teams_records
#             else:
#                 print(f"⚠️ No standings data found for {year}")
#                 return {}
                
#         except Exception as e:
#             print(f"❌ Error loading standings for {year}: {e}")
#             return {}
    
#     # Pull standings for all years
#     all_standings = {}
#     for year in tqdm(years, desc="Loading standings from ESPN"):
#         year_standings = get_standings_from_espn(year)
#         if year_standings:
#             all_standings[year] = year_standings
    
#     if not all_standings:
#         print("ℹ️ No standings data available, using default values (0.500 win%)")
#         games_df['home_team_wins'] = 0
#         games_df['home_team_losses'] = 0
#         games_df['home_team_win_pct'] = 0.5
#         games_df['away_team_wins'] = 0
#         games_df['away_team_losses'] = 0
#         games_df['away_team_win_pct'] = 0.5
#         return games_df
    
#     def get_team_record_on_date(team, date, all_standings):
#         """Get team's record for the season (ESPN provides season totals)"""
#         year = date.year
        
#         # Map team name to ESPN format if needed
#         espn_team = espn_team_mapping.get(team, team)
        
#         if year not in all_standings:
#             return 0, 0, 0.5
            
#         year_standings = all_standings[year]
        
#         # Try both original and mapped team names
#         for team_name in [team, espn_team]:
#             if team_name in year_standings:
#                 record = year_standings[team_name]
#                 return record['wins'], record['losses'], record['win_pct']
        
#         # Default if team not found
#         return 0, 0, 0.5
    
#     # Add team record columns
#     print("Adding team records to game context...")
    
#     # Initialize columns
#     games_df['home_team_wins'] = 0
#     games_df['home_team_losses'] = 0
#     games_df['home_team_win_pct'] = 0.5
#     games_df['away_team_wins'] = 0
#     games_df['away_team_losses'] = 0
#     games_df['away_team_win_pct'] = 0.5
    
#     # Fill in records for each game
#     for idx, row in tqdm(games_df.iterrows(), total=len(games_df), desc="Processing team records"):
#         # Home team record
#         home_wins, home_losses, home_win_pct = get_team_record_on_date(
#             row['home_team'], row['game_date'], all_standings
#         )
#         games_df.loc[idx, 'home_team_wins'] = home_wins
#         games_df.loc[idx, 'home_team_losses'] = home_losses
#         games_df.loc[idx, 'home_team_win_pct'] = home_win_pct
        
#         # Away team record
#         away_wins, away_losses, away_win_pct = get_team_record_on_date(
#             row['away_team'], row['game_date'], all_standings
#         )
#         games_df.loc[idx, 'away_team_wins'] = away_wins
#         games_df.loc[idx, 'away_team_losses'] = away_losses
#         games_df.loc[idx, 'away_team_win_pct'] = away_win_pct
    
#     return games_df

In [4]:
# def get_transformed_data(start_dt, end_dt, weather = True, weather_api_key='VQVKFHQS24N3XBA9HAY2TJZWK'):
#     """
#     Transform Statcast data with optional weather information.
    
#     Parameters:
#     - start_dt, end_dt: Date range for Statcast data
#     - weather_api_key: Optional API key for weather data (Visual Crossing Weather)
#                       If None, weather columns are added but not populated
#                       Get free key at: https://www.visualcrossing.com/weather-api
#     """
#     # Import weather utilities
#     exec(open('weather_utils.py').read())
    
#     df = statcast(start_dt = start_dt, end_dt = end_dt).sort_values(by=['game_date', 'at_bat_number'])
#     minimal_pitch_columns = [
#         'pitch_type',           # What type of pitch
#         'release_speed',        # How fast
#         'plate_x',              # Horizontal location at plate
#         'plate_z',              # Vertical location at plate
#         'pfx_x',                # Horizontal movement 
#         'pfx_z',                # Vertical movement
#         'release_spin_rate',    # Spin rate
#         'zone'                  # Strike zone location
#         ]
    
#     pitch_context_cols = [
#     # Count situation - heavily influences pitch selection
#     'balls',                    # Pre-pitch ball count
#     'strikes',                  # Pre-pitch strike count
    
#     # Game situation
#     'inning',                   # Which inning
#     'inning_topbot',            # Top or bottom of inning
#     'outs_when_up',             # Number of outs
#     'home_score',               # Home team score
#     'away_score',               # Away team score
#     'bat_score_diff',           # Batting team score differential
    
#     # Baserunner situation - affects pitch strategy
#     'on_1b',                    # Runner on 1st base
#     'on_2b',                    # Runner on 2nd base  
#     'on_3b',                    # Runner on 3rd base
    
#     # Player characteristics
#     'stand',                    # Batter handedness (L/R)
#     'p_throws',                 # Pitcher handedness (L/R)
#     'batter',                   # Batter ID (for player-specific tendencies)
#     'pitcher',                  # Pitcher ID (for pitcher-specific tendencies)
    
#     # Pitch sequence context
#     'pitch_number',             # Pitch number in at-bat
#     'at_bat_number',            # At-bat number in game
    
#     # Strike zone reference
#     'sz_top',                   # Top of batter's strike zone
#     'sz_bot',                   # Bottom of batter's strike zone
    
#     # Pitcher workload/fatigue indicators  
#     'n_thruorder_pitcher',      # Times through batting order for pitcher
#     'pitcher_days_since_prev_game',  # Days rest for pitcher
    
#     # Batter recent activity
#     'n_priorpa_thisgame_player_at_bat',  # Prior plate appearances this game
#     'batter_days_since_prev_game',       # Days since batter's last game
    
#     # Age factors (experience)
#     'age_pit',                  # Pitcher age
#     'age_bat',                  # Batter age
    
#     # Defensive positioning
#     'if_fielding_alignment',    # Infield alignment
#     'of_fielding_alignment',    # Outfield alignment
#     'home_team',
#     'away_team',
#     'game_date',
#     'at_bat_number',
#     ]

#     game_context_cols = [
#     'game_type',                # Regular season, playoffs, etc.
#     'home_team',                # Home team
#     'away_team',                # Away team
#     'game_date',                # Date of game
#     ]
    
#     # Create base game context
#     game_context = (
#     df.sort_values(['home_team', 'game_date', 'at_bat_number'])
#       .groupby(['home_team', 'game_date'])[game_context_cols]
#       .first()
#     ).reset_index(drop=True)
    
#     # Ensure game_date is datetime format for team records processing
#     game_context['game_date'] = pd.to_datetime(game_context['game_date'])
    
#     # Add weather data to game context
#     if weather:
#         print(f"Adding weather data to {len(game_context)} games...")
#         game_context = add_weather_to_games(game_context, weather_api_key)
    
#     pitch_context = df[pitch_context_cols]
#     pitch = df[minimal_pitch_columns]
#     play_results = df['events']
#     pitch_results = df['description']
#     home_team_win_target = (
#         df.sort_values(['home_team', 'game_date', 'at_bat_number'])
#         .groupby(['home_team', 'game_date'])['home_win_exp']
#         .last()
#         .pipe(np.round)
#         )
    
#     # Get pitcher historical stats
#     pitcher_stats = get_pitcher_historical_stats(pitchers=df['pitcher'].unique())

#     batter_stats = get_batter_historical_stats(batters=df['batter'].unique())
    
#     # Create pitcher mapping for each game
#     pitcher_by_game = (
#         df.sort_values(['home_team', 'game_date', 'at_bat_number'])
#         .groupby(['home_team', 'game_date'])['pitcher']
#         .first()  # Get the starting pitcher for each game
#         .reset_index()
#     )
    
#     # Ensure both DataFrames have proper datetime format before merge
#     pitcher_by_game['game_date'] = pd.to_datetime(pitcher_by_game['game_date'])
#     game_context['game_date'] = pd.to_datetime(game_context['game_date'])
    
#     # Merge game context with pitcher info
#     game_context = game_context.merge(
#         pitcher_by_game,
#         on=['home_team', 'game_date'],
#         how='left'
#     )
    
#     # Add pitcher stats columns to game context
#     if not pitcher_stats.empty:
#         # Get the column names from pitcher stats (excluding pitcher info columns)
#         pitcher_stat_cols = pitcher_stats.columns.tolist()
        
#         # Create a default row of zeros for pitchers not in historical data
#         default_stats = pd.Series(0.0, index=pitcher_stat_cols)
        
#         # Map each pitcher to their stats, using defaults if not found
#         for col in pitcher_stat_cols:
#             game_context[f'pitcher_{col}'] = game_context['pitcher'].map(
#                 lambda p: pitcher_stats.loc[p, col] if p in pitcher_stats.index else default_stats[col]
#             )
    
#     # Drop the temporary pitcher column
#     game_context.drop(columns=['pitcher'], inplace=True)
    
#     # Add team records data
#     game_context = get_team_records_data(game_context)

#     pitch_context = pitch_context.merge(pitcher_stats, left_on='pitcher', right_index=True, how='left', suffixes=('', '_pit'))
#     pitch_context = pitch_context.merge(batter_stats, left_on='batter', right_index=True, how='left', suffixes=('', '_bat'))

#     pitch_context = pitch_context.drop(columns=['age_pit', 'age_bat', 'if_fielding_alignment', 'of_fielding_alignment', 'oaa'])
#     pitch_context[['on_1b', 'on_2b', 'on_3b']] = pitch_context[['on_1b', 'on_2b', 'on_3b']].fillna(0)
#     pitch_context = pd.get_dummies(pitch_context, columns=pitch_context.select_dtypes(include=['object']).columns, drop_first=True).astype(float)
#     pitch_context = pitch_context.fillna(pitch_context.mean())

#     res_dict = {
#         'game_context': game_context,
#         'pitch_context': pitch_context,
#         'pitch' : pitch,
#         'pitch_result' : pitch_results,
#         'at_bat_target' : play_results,
#         'Game_target' : home_team_win_target.reset_index()['home_win_exp']
#     }   

#     return res_dict

In [5]:
start_dt = '2020-05-12'
end_dt = '2025-08-01'
train = get_transformed_data(start_dt, end_dt, weather=False)
for key in train:
    train[key].to_csv(f'train_{key}_{start_dt}_{end_dt}.csv', index=False)

This is a large query, it may take a moment to complete
Skipping offseason dates
Skipping offseason dates
Skipping offseason dates
Skipping offseason dates
Skipping offseason dates
Skipping offseason dates


Loading standings from ESPN:  33%|███▎      | 2/6 [00:00<00:01,  3.20it/s]

✅ Loaded 30 teams from ESPN for 2020
✅ Loaded 30 teams from ESPN for 2021


Loading standings from ESPN:  50%|█████     | 3/6 [00:00<00:00,  4.09it/s]

✅ Loaded 30 teams from ESPN for 2022
✅ Loaded 30 teams from ESPN for 2023


Loading standings from ESPN: 100%|██████████| 6/6 [00:01<00:00,  4.39it/s]


✅ Loaded 30 teams from ESPN for 2024
✅ Loaded 30 teams from ESPN for 2025
Adding team records to game context...


Processing team records: 100%|██████████| 13211/13211 [00:03<00:00, 4037.11it/s]


In [6]:
start_dt = '2025-08-01'
end_dt = '2025-11-03'
test = get_transformed_data(start_dt, end_dt, weather=False)
for key in test:
    test[key].to_csv(f'test_{key}_{start_dt}_{end_dt}.csv', index=False)

This is a large query, it may take a moment to complete


Loading standings from ESPN: 100%|██████████| 1/1 [00:00<00:00,  2.75it/s]


✅ Loaded 30 teams from ESPN for 2025
Adding team records to game context...


Processing team records: 100%|██████████| 838/838 [00:00<00:00, 3719.17it/s]


In [ ]:
test['pitch_context']

,balls,strikes,inning,outs_when_up,home_score,away_score,bat_score_diff,on_1b,on_2b,on_3b,...,away_team_PHI,away_team_PIT,away_team_SD,away_team_SEA,away_team_SF,away_team_STL,away_team_TB,away_team_TEX,away_team_TOR,away_team_WSH
2024-05-12_BAL_AZ_1_2,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-05-12_BAL_AZ_1_1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-05-12_BOS_WSH_1_1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2024-05-12_CWS_CLE_1_2,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-05-12_CWS_CLE_1_1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-10-01_LAD_CIN_82_2,0.0,1.0,9.0,1.0,8.0,4.0,-4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-10-01_LAD_CIN_82_1,0.0,0.0,9.0,1.0,8.0,4.0,-4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-10-01_LAD_CIN_83_3,0.0,2.0,9.0,2.0,8.0,4.0,-4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-10-01_LAD_CIN_83_2,0.0,1.0,9.0,2.0,8.0,4.0,-4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
dummy_cols = ['game_type', 'home_team', 'away_team', 'conditions'] if 'conditions' in train['game_context'].columns else ['game_type', 'home_team', 'away_team']

train_dummies = pd.get_dummies(train['game_context'], columns=dummy_cols)
test_dummies = pd.get_dummies(test['game_context'], columns=dummy_cols)

train_dummies.index = train['game_context'].index
test_dummies.index = test['game_context'].index

train_dummies.drop(columns=['game_date'], inplace=True)
test_dummies.drop(columns=['game_date'], inplace=True)
train_dummies.fillna(train_dummies.mean(), inplace=True)
test_dummies.fillna(train_dummies.mean(), inplace=True)

test_dummies = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)

train_X = train_dummies.values
test_X = test_dummies.values

In [13]:
train_y = np.array(train['Game_target'])
test_y = np.array(test['Game_target'])

In [14]:
train_X

array([[14.0, 20.0, 12.0, ..., False, False, False],
       [39.0, 32.0, 33.0, ..., False, False, False],
       [0.0, 0.0, 0.0, ..., False, False, False],
       ...,
       [0.0, 0.0, 0.0, ..., False, False, False],
       [36.0, 31.0, 20.0, ..., False, False, False],
       [0.0, 0.0, 0.0, ..., False, False, False]],
      shape=(4415, 90), dtype=object)

In [15]:
from sklearn.neural_network import MLPClassifier

np.random.seed(0)
model = MLPClassifier(hidden_layer_sizes=[1000, 1000, 1000, 1000], learning_rate='adaptive', early_stopping=True)

model.fit(train_X, train_y)
model.score(train_X, train_y), model.score(test_X, test_y)

(0.5714609286523217, 0.5714609286523217)

In [16]:
train_y.mean(), test_y.mean()

(np.float64(0.5270668176670442), np.float64(0.5270668176670442))

In [17]:
pc = train['pitch_context']
pc[['on_1b', 'on_2b', 'on_3b']] = pc[['on_1b', 'on_2b', 'on_3b']].fillna(0)
pc = pd.get_dummies(pc, columns=pc.select_dtypes(include=['object']).columns, drop_first=True).astype(float)
pc = pc.fillna(pc.mean())

In [18]:
pitch_map = {i: pitch for i, pitch in enumerate(train['pitch']['pitch_type'].unique())}
pitch_id = train['pitch']['pitch_type'].map({v: k for k, v in pitch_map.items()})

result_map = {i : result for i, result in enumerate(train['pitch_result'].unique())}
result_id = train['pitch_result'].map({v: k for k, v in result_map.items()})

In [19]:
pitch_data = pd.get_dummies(train['pitch'], drop_first=True).astype(float)
pitch_result = train['pitch_result'].map({v: k for k, v in result_map.items()})

In [20]:
pitch_data.fillna(pitch_data.mean(), inplace=True)

,release_speed,plate_x,plate_z,pfx_x,pfx_z,release_spin_rate,zone,pitch_type_CS,pitch_type_CU,pitch_type_EP,...,pitch_type_FS,pitch_type_KC,pitch_type_KN,pitch_type_PO,pitch_type_SC,pitch_type_SI,pitch_type_SL,pitch_type_ST,pitch_type_SV,pitch_type_UN
2024-05-12_BAL_AZ_1_2,91.3,0.411976,2.678008,-0.81,1.74,2365.0,3.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-05-12_BAL_AZ_1_1,91.3,0.918403,1.491528,-0.73,1.75,2387.0,14.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-05-12_BOS_WSH_1_1,95.1,-0.799422,2.428045,-1.54,0.50,2052.0,4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2024-05-12_CWS_CLE_1_2,91.7,-0.471866,1.808786,-1.12,0.68,2305.0,7.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2024-05-12_CWS_CLE_1_1,92.7,0.811720,2.905910,-0.50,1.34,2501.0,3.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-10-01_LAD_CIN_82_2,100.3,0.440824,2.060265,-0.74,1.21,2109.0,9.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-10-01_LAD_CIN_82_1,100.5,0.054987,1.239123,-0.87,1.15,2108.0,14.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-10-01_LAD_CIN_83_3,99.8,-0.911615,3.232756,-0.64,1.19,1996.0,11.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-10-01_LAD_CIN_83_2,101.4,0.279420,1.767256,-1.09,1.08,2073.0,9.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
from sklearn.neural_network import MLPClassifier

result_model = MLPClassifier(hidden_layer_sizes=[100, 100, 100, 100], learning_rate='adaptive', early_stopping=True)
result_model.fit(pitch_data.iloc[:int(len(pitch_data) * 0.75)], pitch_result.iloc[:int(len(pitch_data) * 0.75)])
result_model.score(pitch_data.iloc[:int(len(pitch_data) * 0.75)], pitch_result.iloc[:int(len(pitch_data) * 0.75)]), result_model.score(pitch_data.iloc[int(len(pitch_data) * 0.75):], pitch_result.iloc[int(len(pitch_data) * 0.75):])

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:792: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


(0.4979855817568579, 0.5005241042096417)

In [ ]:
pitch_result[int(len(pitch_result) * 0.75):].value_counts(normalize=True)

description
1     0.332881
4     0.179821
0     0.174076
2     0.161277
3     0.106074
7     0.021594
8     0.010314
10    0.005739
6     0.003215
5     0.002738
11    0.001748
13    0.000293
9     0.000125
12    0.000067
14    0.000037
Name: proportion, dtype: float64

In [ ]:
train['pitch_result'].value_counts(normalize=True)

description
ball                       3.332674e-01
foul                       1.799873e-01
hit_into_play              1.747967e-01
called_strike              1.620088e-01
swinging_strike            1.051656e-01
blocked_ball               2.082133e-02
foul_tip                   1.011691e-02
swinging_strike_blocked    5.545123e-03
automatic_ball             3.207278e-03
hit_by_pitch               2.769505e-03
foul_bunt                  1.745743e-03
missed_bunt                2.926126e-04
automatic_strike           1.627323e-04
pitchout                   7.563615e-05
bunt_foul_tip              3.590807e-05
intent_ball                7.640015e-07
foul_pitchout              7.640015e-07
Name: proportion, dtype: float64

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

class TransFusionPitchModel(nn.Module):
    def __init__(self, seq_len, context_dim, continuous_dim, num_pitch_types, d_model=128, nhead=8, num_layers=4):
        super(TransFusionPitchModel, self).__init__()
        
        self.seq_len = seq_len
        self.d_model = d_model
        
        input_dim = continuous_dim + num_pitch_types + context_dim
        self.input_proj = nn.Linear(input_dim, d_model)
        
        self.pos_encoder = nn.Parameter(torch.randn(1, seq_len, d_model))
        
        encoder_layers = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        
        self.classifier_head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, num_pitch_types)
        )
    
        self.diffusion_cond_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU()
        )
        
    def forward(self, x, context_features):
        combined_input = torch.cat([x, context_features], dim=-1)
        
        embeddings = self.input_proj(combined_input) + self.pos_encoder
        
        transformer_out = self.transformer_encoder(embeddings)
        
        last_hidden_state = transformer_out[:, -1, :] 
        
        pitch_type_logits = self.classifier_head(last_hidden_state)
        diffusion_condition = self.diffusion_cond_head(last_hidden_state)
        
        return pitch_type_logits, diffusion_condition

class PitchDiffusionNet(nn.Module):
    def __init__(self, continuous_dim, cond_dim, hidden_dim=128):
        super(PitchDiffusionNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(continuous_dim + cond_dim + 1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, continuous_dim)
        )
        
    def forward(self, x_noisy, t, condition):
        t_expanded = t.unsqueeze(-1).float()
        input_vec = torch.cat([x_noisy, condition, t_expanded], dim=-1)
        return self.net(input_vec)

def train_model(model, X, y, epochs=5, batch_size=256, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    model = model.to(device)
    
    # Handle NaNs in y before creating TensorDataset
    valid_idx = ~pd.isna(y)
    X = X[valid_idx]
    y = y[valid_idx]
    
    dataset = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_X, batch_y in loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            
            context = batch_X.unsqueeze(1) 
            
            missing_dim = model.input_proj.in_features - context.shape[-1]
            seq_x = torch.zeros((batch_X.size(0), 1, missing_dim)).to(device)
            
            pitch_type_logits, diffusion_condition = model(seq_x, context)
            
            loss = criterion(pitch_type_logits.squeeze(1), batch_y)
            
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        print(f"Epoch {epoch+1}/{epochs} | Classification Loss: {total_loss/len(loader):.4f}")
        
    return model

pitch_model = TransFusionPitchModel(
    seq_len=3,  # Single pitch input
    context_dim=pc.shape[1],  # Number of context features
    continuous_dim=0,  # No continuous features in this example
    num_pitch_types=len(result_map)  # Number of unique pitch types
)

train_X = pc.values
# train_y = pitch_id.values[:10000]
train_y = result_id.values

pitch_model = train_model(pitch_model, train_X, train_y, epochs=50)

Epoch 1/50 | Classification Loss: 1.7065
Epoch 2/50 | Classification Loss: 1.7128
Epoch 3/50 | Classification Loss: 1.7192
Epoch 4/50 | Classification Loss: 1.7190
Epoch 5/50 | Classification Loss: 1.7189
Epoch 6/50 | Classification Loss: 1.7187


KeyboardInterrupt: 

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
pitch_model = pitch_model.to(device)
pitch_model.eval()

with torch.no_grad():
    valid_idx = ~pd.isna(train_y)
    eval_X = torch.tensor(train_X[valid_idx], dtype=torch.float32).to(device)
    eval_y = torch.tensor(train_y[valid_idx], dtype=torch.long).to(device)
    
    context = eval_X.unsqueeze(1)
    missing_dim = pitch_model.input_proj.in_features - context.shape[-1]
    seq_x = torch.zeros((eval_X.size(0), 1, missing_dim)).to(device)
    
    pitch_type_logits, _ = pitch_model(seq_x, context)
    accuracy = (pitch_type_logits.squeeze(1).argmax(dim=1) == eval_y).float().mean()

print(f"Training Accuracy: {accuracy.item():.4f}")

Training Accuracy: 0.3369


In [172]:
pd.Series(train_y).value_counts(normalize=True)

1     0.3369
4     0.1784
0     0.1733
2     0.1633
3     0.0993
7     0.0235
8     0.0109
10    0.0062
6     0.0035
5     0.0029
11    0.0014
13    0.0002
9     0.0001
12    0.0001
Name: proportion, dtype: float64

In [155]:
train['pitch']

,pitch_type,release_speed,plate_x,plate_z,pfx_x,pfx_z,release_spin_rate,zone
4298,FF,91.3,0.411976,2.678008,-0.81,1.74,2365,3
4358,FF,91.3,0.918403,1.491528,-0.73,1.75,2387,14
4372,SI,95.1,-0.799422,2.428045,-1.54,0.5,2052,4
4244,SI,91.7,-0.471866,1.808786,-1.12,0.68,2305,7
4405,FF,92.7,0.81172,2.90591,-0.5,1.34,2501,3
...,...,...,...,...,...,...,...,...
939,FF,100.3,0.440824,2.060265,-0.74,1.21,2109,9
979,FF,100.5,0.054987,1.239123,-0.87,1.15,2108,14
747,FF,99.8,-0.911615,3.232756,-0.64,1.19,1996,11
785,FF,101.4,0.27942,1.767256,-1.09,1.08,2073,9


4298    field_out
4358          NaN
4372       single
4244    field_out
4405          NaN
          ...    
939           NaN
979           NaN
747     field_out
785           NaN
824           NaN
Name: events, Length: 1308898, dtype: str

In [27]:
train['pitch_context']

,balls,strikes,inning,outs_when_up,home_score,away_score,bat_score_diff,on_1b,on_2b,on_3b,...,away_team_PHI,away_team_PIT,away_team_SD,away_team_SEA,away_team_SF,away_team_STL,away_team_TB,away_team_TEX,away_team_TOR,away_team_WSH
2024-05-12_BAL_AZ,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-05-12_BAL_AZ,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-05-12_BOS_WSH,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2024-05-12_CWS_CLE,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-05-12_CWS_CLE,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-10-01_LAD_CIN,0.0,1.0,9.0,1.0,8.0,4.0,-4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-10-01_LAD_CIN,0.0,0.0,9.0,1.0,8.0,4.0,-4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-10-01_LAD_CIN,0.0,2.0,9.0,2.0,8.0,4.0,-4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-10-01_LAD_CIN,0.0,1.0,9.0,2.0,8.0,4.0,-4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
game_context['game_date']

0      2024-05-21
1      2024-05-22
2      2024-05-23
3      2024-05-24
4      2024-05-25
          ...    
4410   2025-09-16
4411   2025-09-17
4412   2025-09-26
4413   2025-09-27
4414   2025-09-28
Name: game_date, Length: 4415, dtype: datetime64[us]

In [22]:
from xgboost import XGBClassifier

X = train['game_context']


In [47]:
train['pitch'] = train['pitch'].sort_values(['at_bat_id', 'pitch_id'])
result = train['pitch'].groupby('game_id', as_index=False).first().drop(columns=['game_id', 'game_date', 'home_team', 'away_team', 'at_bat_id', 'pitch_id'])

In [54]:
first_pitch_predictor = MLPClassifier(hidden_layer_sizes=[100, 100], learning_rate='adaptive', early_stopping=True)
first_pitch_X = train['game_context'].drop(columns=train['game_context'].select_dtypes(include=['object']).columns).values
first_pitch_y = pd.get_dummies(result, columns=result.select_dtypes(include=['object']).columns, drop_first=True).values
first_pitch_predictor.fit(first_pitch_X, first_pitch_y)

/var/folders/gf/ct6w19qn6874rwzwymj7mht80000gn/T/ipykernel_57328/1066896708.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  first_pitch_X = train['game_context'].drop(columns=train['game_context'].select_dtypes(include=['object']).columns).values
/var/folders/gf/ct6w19qn6874rwzwymj7mht80000gn/T/ipykernel_57328/1066896708.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to

TypeError: float() argument must be a string or a real number, not 'Timestamp'

In [44]:
train['game_context'].sort_values(['game_date', 'home_team'], inplace=True)
train['pitch'].sort_values(['game_date', 'home_team'], inplace=True)

In [56]:
for col in train['game_context'].columns:
    print(f"{col}: {train['game_context'][col].dtype}")

game_type: str
home_team: str
away_team: str
game_date: datetime64[us]
pitcher_xwoba: float64
pitcher_xba: float64
pitcher_xslg: float64
pitcher_xiso: float64
pitcher_xobp: float64
pitcher_brl: float64
pitcher_brl_percent: float64
pitcher_exit_velocity: float64
pitcher_max_ev: float64
pitcher_hard_hit_percent: float64
pitcher_k_percent: float64
pitcher_bb_percent: float64
pitcher_whiff_percent: float64
pitcher_chase_percent: float64
pitcher_xera: float64
pitcher_fb_velocity: float64
pitcher_fb_spin: float64
pitcher_curve_spin: float64
home_team_wins: int64
home_team_losses: int64
home_team_win_pct: float64
away_team_wins: int64
away_team_losses: int64
away_team_win_pct: float64


In [61]:
for col in train['pitch_context'].columns:
    print(f"{col}")

balls
strikes
inning
outs_when_up
home_score
away_score
bat_score_diff
on_1b
on_2b
on_3b
batter
pitcher
pitch_number
at_bat_number
sz_top
sz_bot
n_thruorder_pitcher
pitcher_days_since_prev_game
n_priorpa_thisgame_player_at_bat
batter_days_since_prev_game
xwoba
xba
xslg
xiso
xobp
brl
brl_percent
exit_velocity
max_ev
hard_hit_percent
k_percent
bb_percent
whiff_percent
chase_percent
xera
fb_velocity
fb_spin
curve_spin
xwoba_bat
xba_bat
xslg_bat
xiso_bat
xobp_bat
brl_bat
brl_percent_bat
exit_velocity_bat
max_ev_bat
hard_hit_percent_bat
k_percent_bat
bb_percent_bat
whiff_percent_bat
chase_percent_bat
arm_strength
sprint_speed
bat_speed
squared_up_rate
swing_length
inning_topbot_Top
stand_R
p_throws_R
home_team_ATL
home_team_AZ
home_team_BAL
home_team_BOS
home_team_CHC
home_team_CIN
home_team_CLE
home_team_COL
home_team_CWS
home_team_DET
home_team_HOU
home_team_KC
home_team_LAA
home_team_LAD
home_team_MIA
home_team_MIL
home_team_MIN
home_team_NYM
home_team_NYY
home_team_PHI
home_team_PIT
hom

In [12]:
train['first_pitch']

,pitch_type,release_speed,plate_x,plate_z,pfx_x,pfx_z,release_spin_rate,zone,game_date,home_team,away_team,at_bat_id,pitch_id
game_id,,,,,,,,,,,,,
2024-05-12_BAL_AZ,FF,91.3,0.918403,1.491528,-0.73,1.75,2387,14,2024-05-12,BAL,AZ,1,1
2024-05-12_BOS_WSH,SI,95.1,-0.799422,2.428045,-1.54,0.5,2052,4,2024-05-12,BOS,WSH,1,1
2024-05-12_COL_TEX,SI,88.4,1.132679,1.853721,1.08,0.31,2049,14,2024-05-12,COL,TEX,1,1
2024-05-12_CWS_CLE,FF,92.7,0.81172,2.90591,-0.5,1.34,2501,3,2024-05-12,CWS,CLE,1,1
2024-05-12_DET_HOU,FF,93.4,-1.230848,2.442286,-0.26,1.19,2197,11,2024-05-12,DET,HOU,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-30_NYY_BOS,FC,94.9,0.079546,2.314397,-0.19,0.75,2285,5,2025-09-30,NYY,BOS,1,1
2025-10-01_CHC_SD,SL,91.2,-0.337626,1.33918,0.21,0.44,2606,13,2025-10-01,CHC,SD,1,1
2025-10-01_CLE_DET,FC,85.5,-0.250373,2.486901,0.91,-0.21,2545,5,2025-10-01,CLE,DET,1,1


In [44]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

class first_pitch_predictor(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_pitches, n_zones, n_targets):
        super(first_pitch_predictor, self).__init__()
        self.pitch_type_model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_pitches),
        )

        self.strike_zone_model = nn.Sequential(
            nn.Linear(input_dim + n_pitches, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_zones),
            nn.Softmax(dim=-1)
        )

        self.pitch_model = nn.Sequential(
            nn.Linear(input_dim + n_pitches + n_zones, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_targets)
        )

    def predict_pitch_type(self, x):
        return self.pitch_type_model(x)
    
    def predict_strike_zone(self, x, pitch_type):
        combined = torch.cat([x, pitch_type], dim=-1)
        return self.strike_zone_model(combined)
    
    def predict_pitch(self, x, pitch_type, strike_zone):
        combined = torch.cat([x, pitch_type, strike_zone], dim=-1)
        return self.pitch_model(combined)

    def forward(self, x):
        pitch_type_logits = self.predict_pitch_type(x)
        pitch_type_probs = torch.softmax(pitch_type_logits, dim=-1)
        strike_zone_probs = self.predict_strike_zone(x, pitch_type_probs)
        pitch_prediction = self.predict_pitch(x, pitch_type_probs, strike_zone_probs)
        return pitch_type_probs, strike_zone_probs, pitch_prediction
    
def train_first_pitch_model(model, X, y_pitch_type, y_strike_zone, y_pitch, epochs=10, batch_size=256, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    model = model.to(device)
    
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32), 
        torch.tensor(y_pitch_type, dtype=torch.long), 
        torch.tensor(y_strike_zone, dtype=torch.long), 
        torch.tensor(y_pitch, dtype=torch.float32)
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    criterion_pitch_type = nn.CrossEntropyLoss()
    criterion_strike_zone = nn.CrossEntropyLoss()
    criterion_pitch = nn.MSELoss()
    
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_X, batch_y_pt, batch_y_sz, batch_y_p in loader:
            batch_X, batch_y_pt, batch_y_sz, batch_y_p = batch_X.to(device), batch_y_pt.to(device), batch_y_sz.to(device), batch_y_p.to(device)
            
            optimizer.zero_grad()
            
            pitch_type_logits, strike_zone_probs, pitch_prediction = model(batch_X)
            
            loss_pt = criterion_pitch_type(pitch_type_logits, batch_y_pt)
            loss_sz = criterion_strike_zone(strike_zone_probs.view(-1, strike_zone_probs.size(-1)), batch_y_sz)
            loss_p = criterion_pitch(pitch_prediction.view(-1, pitch_prediction.size(-1)), batch_y_p)
            
            loss = loss_pt + loss_sz + loss_p
            
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(loader):.4f}")
        
    return model

if __name__ == "__main__":
    X = pd.get_dummies(train['game_context'].drop(columns=['game_date', 'game_id'])).astype(float).values
    y = train['first_pitch'].drop(columns=['home_team', 'away_team', 'game_date', 'at_bat_id', 'pitch_id'])

    y_pitch_type = pd.get_dummies(y['pitch_type'], drop_first=True).values
    y_strike_zone = pd.get_dummies(y['zone'], drop_first=True).values
    y_pitch = y.drop(columns=['pitch_type', 'zone']).values

    num_zones = y['zone'].nunique()
    num_pitch_types = y['pitch_type'].nunique()

    model = first_pitch_predictor(input_dim=X.shape[1],
                                hidden_dim=128,
                                n_pitches=num_pitch_types,
                                n_targets=y_pitch.shape[1],
                                n_zones = num_zones)


    train_first_pitch_model(model, X, y_pitch_type, y_strike_zone, y_pitch, epochs=20)
    torch.save(model.state_dict(), 'first_pitch_predictor.pth')

In [5]:
X = pd.get_dummies(train['game_context'].drop(columns=['game_date', 'game_id'])).astype(float).values
y = train['first_pitch']

y_pitch_type = pd.get_dummies(y['pitch_type'], drop_first=True).values
y_strike_zone = pd.get_dummies(y['zone'], drop_first=True).values
y_pitch = y.drop(columns=['pitch_type', 'zone']).values

In [ ]:
y

,pitch_type,release_speed,plate_x,plate_z,pfx_x,pfx_z,release_spin_rate,zone
game_id,,,,,,,,
2024-05-12_BAL_AZ,FF,91.3,0.918403,1.491528,-0.73,1.75,2387,14
2024-05-12_BOS_WSH,SI,95.1,-0.799422,2.428045,-1.54,0.5,2052,4
2024-05-12_COL_TEX,SI,88.4,1.132679,1.853721,1.08,0.31,2049,14
2024-05-12_CWS_CLE,FF,92.7,0.81172,2.90591,-0.5,1.34,2501,3
2024-05-12_DET_HOU,FF,93.4,-1.230848,2.442286,-0.26,1.19,2197,11
...,...,...,...,...,...,...,...,...
2025-09-30_NYY_BOS,FC,94.9,0.079546,2.314397,-0.19,0.75,2285,5
2025-10-01_CHC_SD,SL,91.2,-0.337626,1.33918,0.21,0.44,2606,13
2025-10-01_CLE_DET,FC,85.5,-0.250373,2.486901,0.91,-0.21,2545,5


: 

In [ ]:
from pitch_data import get_transformed_data

train = get_transformed_data('2024-05-12', '2025-10-01', weather=False)
test = get_transformed_data('2025-10-01', '2025-11-03', weather=False)

This is a large query, it may take a moment to complete
Skipping offseason dates


Loading standings from ESPN: 100%|██████████| 2/2 [00:00<00:00,  3.74it/s]


✅ Loaded 30 teams from ESPN for 2024
✅ Loaded 30 teams from ESPN for 2025
Adding team records to game context...


Processing team records: 100%|██████████| 4415/4415 [00:01<00:00, 3942.37it/s]


This is a large query, it may take a moment to complete
Skipping offseason dates


 32%|███▏      | 126/389 [00:37<00:28,  9.10it/s]

In [14]:
import pandas as pd

train = {}
test = {}
train['game_context'] = pd.read_csv('train_game_context_2024-05-12_2025-10-01.csv')
train['first_pitch'] = pd.read_csv('train_first_pitch_2024-05-12_2025-10-01.csv')
X = pd.get_dummies(train['game_context'].drop(columns=['game_date', 'game_id'])).astype(float).values
y = train['first_pitch']
y['pitch_type'] = y['pitch_type'].fillna(y['pitch_type'].mode()[0])
y.fillna(y.groupby('pitch_type').transform('mean'), inplace=True)

y_pitch_type = y['pitch_type'].astype('category').cat.codes.values
y_strike_zone = y['zone'].astype('category').cat.codes.values
y_pitch = y.drop(columns=['pitch_type', 'zone']).values

num_zones = y['zone'].nunique()
num_pitch_types = y['pitch_type'].nunique()

# model = first_pitch_predictor(input_dim=X.shape[1],
#                             hidden_dim=128,
#                             n_pitches=num_pitch_types,
#                             n_targets=y_pitch.shape[1],
#                             n_zones = num_zones)


# train_first_pitch_model(model, X, y_pitch_type, y_strike_zone, y_pitch, epochs=20)
# torch.save(model.state_dict(), 'first_pitch_predictor.pth')

In [15]:
from first_pitch_predictor import first_pitch_predictor, train_first_pitch_model

model = first_pitch_predictor(input_dim=X.shape[1],
                                hidden_dim=128,
                                n_pitches=num_pitch_types,
                                n_targets=y_pitch.shape[1],
                                n_zones = num_zones)

In [16]:
train_first_pitch_model(model, X, y_pitch_type, y_strike_zone, y_pitch, epochs=20)

Epoch 1/20 | Loss: 794743.4306
Epoch 2/20 | Loss: 368523.9227
Epoch 3/20 | Loss: 63388.2476
Epoch 4/20 | Loss: 38710.5201
Epoch 5/20 | Loss: 28797.0352
Epoch 6/20 | Loss: 23649.7738
Epoch 7/20 | Loss: 20570.5033
Epoch 8/20 | Loss: 18203.8047
Epoch 9/20 | Loss: 16940.5401
Epoch 10/20 | Loss: 16402.4534
Epoch 11/20 | Loss: 15564.0114
Epoch 12/20 | Loss: 14899.6614
Epoch 13/20 | Loss: 14295.8705
Epoch 14/20 | Loss: 13628.7975
Epoch 15/20 | Loss: 13105.8558
Epoch 16/20 | Loss: 13129.6405
Epoch 17/20 | Loss: 12700.3286
Epoch 18/20 | Loss: 12538.5220
Epoch 19/20 | Loss: 12517.6185
Epoch 20/20 | Loss: 11989.6130


first_pitch_predictor(
  (pitch_type_model): Sequential(
    (0): Linear(in_features=131, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=11, bias=True)
  )
  (strike_zone_model): Sequential(
    (0): Linear(in_features=142, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=14, bias=True)
  )
  (pitch_model): Sequential(
    (0): Linear(in_features=156, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=6, bias=True)
  )
)

In [13]:
most_common_pitch

'FF'

In [8]:
import pandas as pd

train = {}
test = {}
train['game_context'] = pd.read_csv('train_game_context_2020-05-12_2025-08-01.csv')
train['first_pitch'] = pd.read_csv('train_first_pitch_2020-05-12_2025-08-01.csv')
test['game_context'] = pd.read_csv('test_game_context_2025-08-01_2025-11-03.csv')
test['first_pitch'] = pd.read_csv('test_first_pitch_2025-08-01_2025-11-03.csv')
X = pd.get_dummies(train['game_context'].drop(columns=['game_date', 'game_id'])).astype(float).values
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_std[X_std < 1e-8] = 1.0
X = (X - X_mean) / X_std
y = train['first_pitch']
y['pitch_type'] = y['pitch_type'].fillna(y['pitch_type'].mode()[0])
y.fillna(y.groupby('pitch_type').transform('mean'), inplace=True)

y_pitch_type = y['pitch_type'].astype('category').cat.codes.values
y_strike_zone = y['zone'].astype('category').cat.codes.values
y_pitch = y.drop(columns=['pitch_type', 'zone']).values

# y_pitch_tensor = torch.tensor(y_pitch, dtype=torch.float32)
# pitch_mean = y_pitch_tensor.mean(dim=0)
# pitch_std = y_pitch_tensor.std(dim=0).clamp(min=1e-8)
# y_pitch_normalized = (y_pitch_tensor - pitch_mean) / pitch_std

# num_zones = y['zone'].nunique()
# num_pitch_types = y['pitch_type'].nunique()

In [32]:
train = {}
test = {}
train['game_context'] = pd.read_csv('train_game_context_2020-05-12_2025-08-01.csv')
train['first_pitch'] = pd.read_csv('train_first_pitch_2020-05-12_2025-08-01.csv')
test['game_context'] = pd.read_csv('test_game_context_2025-08-01_2025-11-03.csv')
test['first_pitch'] = pd.read_csv('test_first_pitch_2025-08-01_2025-11-03.csv')
X = pd.get_dummies(train['game_context'].drop(columns=['game_date', 'game_id']), drop_first=True).astype(float).values
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_std[X_std < 1e-8] = 1.0
X = (X - X_mean) / X_std
y = train['first_pitch']
y['pitch_type'] = y['pitch_type'].fillna(y['pitch_type'].mode()[0])
y.fillna(y.groupby('pitch_type').transform('mean'), inplace=True)

y_pitch_type = y['pitch_type'].astype('category').cat.codes.values
y_strike_zone = y['zone'].astype('category').cat.codes.values
y_pitch = y.drop(columns=['pitch_type', 'zone']).values

y_pitch[np.isnan(y_pitch)] = np.nanmean(y_pitch)

num_zones = y['zone'].nunique()
num_pitch_types = y['pitch_type'].nunique()

test_X = pd.get_dummies(test['game_context'].drop(columns=['game_date', 'game_id']), drop_first=True).astype(float).values
test_X = (test_X - X_mean) / X_std
test_y = test['first_pitch']
test_y['pitch_type'] = test_y['pitch_type'].fillna(test_y['pitch_type'].mode()[0])
test_y.fillna(test_y.groupby('pitch_type').transform('mean'), inplace=True)
test_y_pitch_type = test_y['pitch_type'].astype('category').cat.codes.values
test_y_strike_zone = test_y['zone'].astype('category').cat.codes.values
test_y_pitch = test_y.drop(columns=['pitch_type', 'zone']).values
test_y_pitch_tensor = torch.tensor(test_y_pitch, dtype=torch.float32)
test_y_pitch_normalized = (test_y_pitch_tensor - pitch_mean) / pitch_std

acc = get_val_error(model, test_X, test_y_pitch_type, test_y_strike_zone, test_y_pitch_normalized)
print(f"Validation Accuracy - Pitch Type: {acc[0]:.4%}, Strike Zone: {acc[1]:.4%}")
for i, col in enumerate(test_y.drop(columns=['pitch_type', 'zone']).columns):
    print(f"MAE for {col}: {acc[4][i]:.4f}")
print(f"Validation Loss - Pitch Type: {acc[2]:.4f}, Strike Zone: {acc[3]:.4f}")

ValueError: operands could not be broadcast together with shapes (838,127) (128,) 

In [34]:
train_cols = pd.get_dummies(train['game_context'].drop(columns=['game_date', 'game_id']), drop_first=True).columns
test_cols = pd.get_dummies(test['game_context'].drop(columns=['game_date', 'game_id']), drop_first=True).columns

In [38]:
set(train_cols) - set(test_cols), set(test_cols) - set(train_cols)

({'game_type_S'}, set())

In [45]:
test['first_pitch']['pitch_type'].value_counts(normalize=True)

pitch_type
FF    0.711217
SI    0.215990
FC    0.032220
SL    0.017900
ST    0.007160
CH    0.005967
CU    0.004773
KC    0.003580
FS    0.001193
Name: proportion, dtype: float64

In [ ]:
import torch
import pandas as pd
import numpy as np
from first_pitch_predictor import first_pitch_predictor, get_val_error

train = {}
test = {}
train['game_context'] = pd.read_csv('train_game_context_2020-05-12_2025-08-01.csv')
train['first_pitch'] = pd.read_csv('train_first_pitch_2020-05-12_2025-08-01.csv')
test['game_context'] = pd.read_csv('test_game_context_2025-08-01_2025-11-03.csv')
test['first_pitch'] = pd.read_csv('test_first_pitch_2025-08-01_2025-11-03.csv')

# --- Prepare training data (needed for category mappings and normalization stats) ---
X_df = pd.get_dummies(train['game_context'].drop(columns=['game_date', 'game_id']), drop_first=True).astype(float)
train_columns = X_df.columns
X = X_df.values
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_std[X_std < 1e-8] = 1.0

y = train['first_pitch'].copy()
y['pitch_type'] = y['pitch_type'].fillna(y['pitch_type'].mode()[0])
y.fillna(y.groupby('pitch_type').transform('mean'), inplace=True)

pitch_type_categories = pd.CategoricalDtype(categories=sorted(y['pitch_type'].unique()))
zone_categories = pd.CategoricalDtype(categories=sorted(y['zone'].dropna().unique()))
y_pitch = y.drop(columns=['pitch_type', 'zone']).values.copy()
y_pitch[np.isnan(y_pitch)] = np.nanmean(y_pitch)

y_pitch_tensor = torch.tensor(y_pitch, dtype=torch.float32)
pitch_mean = y_pitch_tensor.mean(dim=0)
pitch_std = y_pitch_tensor.std(dim=0).clamp(min=1e-8)

num_zones = len(zone_categories.categories)
num_pitch_types = len(pitch_type_categories.categories)

# --- Prepare test data ---
test_X_df = pd.get_dummies(test['game_context'].drop(columns=['game_date', 'game_id']), drop_first=True).astype(float)
test_X_df = test_X_df.reindex(columns=train_columns, fill_value=0)
test_X = test_X_df.values
test_X = (test_X - X_mean) / X_std
test_y = test['first_pitch'].copy()
test_y['pitch_type'] = test_y['pitch_type'].fillna(test_y['pitch_type'].mode()[0])
test_y.fillna(test_y.groupby('pitch_type').transform('mean'), inplace=True)
test_y_pitch_type = test_y['pitch_type'].astype(pitch_type_categories).cat.codes.values
test_y_strike_zone = test_y['zone'].astype(zone_categories).cat.codes.values
test_y_pitch = test_y.drop(columns=['pitch_type', 'zone']).values
test_y_pitch_tensor = torch.tensor(test_y_pitch, dtype=torch.float32)
test_y_pitch_normalized = (test_y_pitch_tensor - pitch_mean) / pitch_std

# --- Load pretrained model ---
model = first_pitch_predictor(input_dim=X.shape[1],
                            hidden_dim=256,
                            n_pitches=num_pitch_types,
                            n_targets=y_pitch.shape[1],
                            n_zones=num_zones)
model.load_state_dict(torch.load('first_pitch_predictor.pth', weights_only=True))
print("Loaded pretrained weights from first_pitch_predictor.pth")

# --- Evaluate ---
acc = get_val_error(model, test_X, test_y_pitch_type, test_y_strike_zone, test_y_pitch_normalized.numpy())
print(f"Pitch Type Acc: {acc[0]:.2%}, Strike Zone Acc: {acc[1]:.2%}, Cont NLL: {acc[4]:.4f}")

# --- Sample first pitches for test games ---
device = next(model.parameters()).device
test_X_tensor = torch.tensor(test_X, dtype=torch.float32).to(device)
samples = model.sample(test_X_tensor, n_samples=5)

# Decode categories
pitch_type_names = pitch_type_categories.categories
zone_names = zone_categories.categories
continuous_cols = y.drop(columns=['pitch_type', 'zone']).columns.tolist()

for i, s in enumerate(samples):
    print(f"\n--- Sample {i+1} ---")
    sample_df = pd.DataFrame({
        'pitch_type': [pitch_type_names[idx] for idx in s['pitch_type'].cpu().numpy()],
        'zone': [zone_names[idx] for idx in s['zone'].cpu().numpy()],
    })
    # Unnormalize continuous values back to original scale
    continuous_raw = s['continuous'].cpu() * pitch_std + pitch_mean
    for j, col in enumerate(continuous_cols):
        sample_df[col] = continuous_raw[:, j].numpy()
    print(sample_df.head(10))
    print(f"Pitch type distribution: {sample_df['pitch_type'].value_counts(normalize=True).to_dict()}")

ValueError: assignment destination is read-only